# Intent Extraction — Experiment Notebook

**Structură:**
```
[0] Imports & setup        — rulează o singură dată la început
[1] Config experiment      — modifici aici modelul, limba, versiunea promptului
[2] Preview prompt         — vezi promptul randat înainte să cheltuiești API calls
[3] Run experiment         — rulează modelul pe dataset
[4] Rezultate              — tabel predicții + metrici sumar
[5] Salvare                — salvează experiment în outputs/
[6] Comparație experimente — compară două experimente anterioare
```

**Flux tipic pentru o iterație de prompt:**
1. Modifici fișierul `.jinja` în `prompts/intent_extraction/`
2. Schimbi `PROMPT_VERSION` în celula [1]
3. Rulezi [1] → [2] → [3] → [4] → [5]
4. Compari cu versiunea anterioară în [6]

---
## [0] Imports & Setup
*Rulează o singură dată la pornirea notebook-ului.*

In [1]:
import os

os.chdir(
    r"C:\Users\Matebook 14s\Documents"
    r"\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-"
)

print("Working directory:", os.getcwd())

Working directory: C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-


In [2]:
import json
import time
import re
from pathlib import Path
from collections import defaultdict

import numpy as np
from jinja2 import Environment, FileSystemLoader
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score
from tabulate import tabulate

# ── Paths ──────────────────────────────────────────────────────────────────
PROMPTS_DIR  = Path("prompts/intent_extraction")
CONFIGS_DIR  = Path("configs")
DATASET_PATH = Path("data/master_dataset_refined_180.json")
OUTPUT_DIR = Path(r"C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Labels ─────────────────────────────────────────────────────────────────
with open(CONFIGS_DIR /"intent_definitions.json", encoding="utf-8") as f:
    _defs = json.load(f)
LABELS = [l["name"] for l in _defs["labels"]]
VALID_LABELS = set(LABELS)

# ── Dataset ────────────────────────────────────────────────────────────────
with open(DATASET_PATH, encoding="utf-8") as f:
    DATASET = json.load(f)["conversations"]

print(f"✓ Dataset: {len(DATASET)} conversații")
print(f"✓ Labels:  {len(LABELS)} intenții")
print(f"✓ Output:  {OUTPUT_DIR}")

✓ Dataset: 180 conversații
✓ Labels:  12 intenții
✓ Output:  C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\outputs


---
## [0b] Funcții utilitare
*Rulează o singură dată după [0].*

In [3]:
# ── Prompt renderer ────────────────────────────────────────────────────────
import json
from jinja2 import Environment, FileSystemLoader

with open(
    r"C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\configs\few_shot_examples.json",
    encoding="utf-8"
) as f:
    few_shot_examples = json.load(f)

env = Environment(loader=FileSystemLoader(
    r"C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\prompts\intent_extraction"
))

def render_prompt(lang: str, turns: list, version: str = "v1") -> str:
    template = env.get_template(f"{lang}_zero_shot_{version}.jinja")
    return template.render(
        conversation=turns,
        examples=few_shot_examples if version == "v4" else []
    )

# ── Response parser ────────────────────────────────────────────────────────
def parse_response(raw: str):
    """
    Returnează (intent, confidence, reasoning, parse_failed).
    Gestionează JSON înfășurat în ```json ... ```.
    """
    text = raw.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    try:
        data = json.loads(text)
        intent = data.get("intent")
        return (
            intent,
            data.get("confidence"),
            data.get("reasoning"),
            intent not in VALID_LABELS,
        )
    except json.JSONDecodeError:
        return None, None, None, True

# ── Metrici ────────────────────────────────────────────────────────────────
def compute_metrics(y_true, y_pred):
    if not y_true:
        return {}
    return {
        "accuracy":    round(accuracy_score(y_true, y_pred), 4),
        "macro_f1":    round(f1_score(y_true, y_pred, average="macro",    labels=LABELS, zero_division=0), 4),
        "weighted_f1": round(f1_score(y_true, y_pred, average="weighted", labels=LABELS, zero_division=0), 4),
        "kappa":       round(cohen_kappa_score(y_true, y_pred, labels=LABELS), 4),
        "n_valid":     len(y_true),
    }

# ── Tabel predicții ────────────────────────────────────────────────────────
def print_predictions_table(results: list):
    rows = []
    for r in results:
        if r["parse_failed"]:
            mark = "FAIL"
        elif r["predicted_intent"] == r["dataset_label"]:
            mark = "✓"
        else:
            mark = f"✗  →  {r['predicted_intent']}"
        rows.append([
            r["conversation_id"],
            r["dataset_label"],
            mark,
            r.get("confidence", "—"),
            f"{r['latency_ms']:.0f}ms",
        ])
    print(tabulate(rows,
                   headers=["conv_id", "dataset_label", "predicție", "confidence", "latență"],
                   tablefmt="rounded_outline"))

# ── Tabel metrici sumar ────────────────────────────────────────────────────
def print_metrics_summary(results: list, exp_name: str):
    y_true  = [r["dataset_label"]    for r in results if not r["parse_failed"]]
    y_pred  = [r["predicted_intent"] for r in results if not r["parse_failed"]]
    lats    = [r["latency_ms"]       for r in results if r["latency_ms"] > 0]
    fails   = sum(1 for r in results if r["parse_failed"])
    total   = len(results)

    m = compute_metrics(y_true, y_pred)

    rows = [
        ["Experiment",    exp_name],
        ["Accuracy",      f"{m.get('accuracy', 0)*100:.1f}%"],
        ["Macro F1",      f"{m.get('macro_f1', 0)*100:.1f}%"],
        ["Weighted F1",   f"{m.get('weighted_f1', 0)*100:.1f}%"],
        ["Cohen κ",       f"{m.get('kappa', 0):.3f}"],
        ["Parse failures",f"{fails}/{total}  ({fails/total*100:.1f}%)"],
        ["Latență medie", f"{np.mean(lats):.0f}ms" if lats else "—"],
        ["Latență p95",   f"{np.percentile(lats, 95):.0f}ms" if lats else "—"],
        ["N valid",       len(y_true)],
    ]
    print(tabulate(rows, tablefmt="simple", headers=["Metrică", "Valoare"]))
    return m

# ── Top erori ──────────────────────────────────────────────────────────────
def print_top_errors(results: list, n: int = 10):
    errors = defaultdict(lambda: defaultdict(int))
    for r in results:
        if not r["parse_failed"] and r["predicted_intent"] != r["dataset_label"]:
            errors[r["dataset_label"]][r["predicted_intent"]] += 1

    rows = [
        [true, pred, cnt]
        for true, preds in errors.items()
        for pred, cnt in preds.items()
    ]
    rows.sort(key=lambda r: -r[2])
    if rows:
        print(tabulate(rows[:n],
                       headers=["Intenție reală", "Prezis ca", "# greșeli"],
                       tablefmt="simple"))
    else:
        print("  0 erori.")

print(f"✓ Funcții utilitare încărcate. {len(few_shot_examples)} exemple few-shot.")

✓ Funcții utilitare încărcate. 6 exemple few-shot.


---
## [1] Config experiment
**Modifici doar această celulă când schimbi modelul, limba sau versiunea promptului.**

In [4]:
# ═══════════════════════════════════════════════════════════
#  CONFIGURAȚIE EXPERIMENT — editează aici
# ═══════════════════════════════════════════════════════════

MODEL         = "roberta_encoder"       # openai_o3 | gemini_2.5_flash | aya_expanse_8b | rollama2_7b | robert_encoder | roberta_encoder
LANG          = "en"              # ro | en
PROMPT_VERSION = "v4"             # v1 | v2 | v3 | ...

# Rulează pe tot datasetul sau doar pe primele N conversații (None = tot)
MAX_CONVERSATIONS = None

# ═══════════════════════════════════════════════════════════
EXP_NAME = f"{MODEL}__{LANG}__{PROMPT_VERSION}"
EXP_FILE = OUTPUT_DIR / f"exp_{EXP_NAME}.json"

conversations = DATASET[:MAX_CONVERSATIONS] if MAX_CONVERSATIONS else DATASET

print(f"Experiment : {EXP_NAME}")
print(f"Model      : {MODEL}")
print(f"Limbă      : {LANG}")
print(f"Prompt     : {LANG}_zero_shot_{PROMPT_VERSION}.jinja")
print(f"Dataset    : {len(conversations)} conversații")
print(f"Output     : {EXP_FILE}")

Experiment : roberta_encoder__en__v4
Model      : roberta_encoder
Limbă      : en
Prompt     : en_zero_shot_v4.jinja
Dataset    : 180 conversații
Output     : C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\outputs\exp_roberta_encoder__en__v4.json


---
## [2] Preview prompt
*Verifică promptul randat pe prima conversație înainte să rulezi tot datasetul.*

In [6]:
sample_conv = conversations[0]
sample_prompt = render_prompt(LANG, sample_conv["turns"], PROMPT_VERSION)

print(f"── Conversație: {sample_conv['conversation_id']}")
print(f"── dataset_label: {sample_conv['intent']}")
print("─" * 60)
print(sample_prompt)

── Conversație: conv_simple_0001
── dataset_label: update_personal_data
────────────────────────────────────────────────────────────
Developer: # Role and Objective
You are an intent classification assistant for a Romanian banking voicebot system.

Your objective is to identify the **primary intent** of the entire conversation below. The intent is determined by what the user wanted to accomplish within the conversation.

# Intent Labels
- **`block_card`**: The user wants to block a lost, stolen, retained, or compromised card.
- **`unblock_card`**: The user wants to unblock a previously blocked card.
- **`open_account`**: The user wants to open a new bank account.
- **`close_account`**: The user wants to close an existing bank account.
- **`check_balance`**: The user wants to check the account balance or available funds.
- **`get_account_statement`**: The user wants to obtain the account statement or transaction history.
- **`report_suspicious_transaction`**: The user reports a suspicio

---
## [3] Inițializare model
*Rulează o singură dată per sesiune pentru fiecare model.*

In [7]:
from dotenv import load_dotenv
load_dotenv()

True

In [8]:
# Inițializează modelul selectat în [1]
# ─────────────────────────────────────────────────────────────────────────
# Fiecare bloc if/elif inițializează clientul corespunzător.
# ─────────────────────────────────────────────────────────────────────────

def _extract_conversation_text(prompt: str) -> str:
    """
    Extrage replicile conversației din prompt.
    Funcționează cu toate versiunile de prompt (v1-v3),
    indiferent dacă secțiunile folosesc # sau ##.
    """
    lines = prompt.split("\n")
    conv_lines, in_conv = [], False
    for line in lines:
        stripped = line.strip()
        if "Convers" in stripped and stripped.startswith("#"):
            in_conv = True
            continue
        if in_conv and stripped.startswith("#"):
            break
        if in_conv and stripped:
            conv_lines.append(stripped)
    return " ".join(conv_lines)


if MODEL == "openai_o3":
    # OpenAI o3 — accesat via API oficial.
    # Necesită OPENAI_API_KEY în variabilele de mediu.
    from openai import OpenAI
    _client = OpenAI()

    def call_model(prompt: str) -> str:
        r = _client.chat.completions.create(
            model="o3",
            messages=[{"role": "user", "content": prompt}],
        )
        return r.choices[0].message.content

elif MODEL == "gemini_2.5_flash":
    import os
    from google.genai import Client
    _client = Client(api_key=os.environ["GEMINI_API_KEY"])

    def call_model(prompt: str) -> str:
        response = _client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt,
        )
        return response.text

elif MODEL == "aya_expanse_8b":
    # Aya Expanse 8B — rulat local via Ollama.
    # Asigură-te că Ollama rulează și modelul e descărcat:
    #   ollama pull aya-expanse:8b
    import ollama as _ollama

    def call_model(prompt: str) -> str:
        r = _ollama.chat(
            model="aya-expanse:8b",
            messages=[{"role": "user", "content": prompt}],
        )
        return r["message"]["content"]

elif MODEL == "rollama2_7b":
    # RoLLaMA 2 7B Instruct — rulat local via Ollama.
    # Asigură-te că Ollama rulează și modelul e descărcat:
    #   ollama pull rollama2:7b-instruct
    # Verifică tag-ul exact cu: ollama list
    import ollama as _ollama

    def call_model(prompt: str) -> str:
        r = _ollama.chat(
            model="rollama2:7b-instruct",
            messages=[{"role": "user", "content": prompt}],
        )
        return r["message"]["content"]

elif MODEL == "roberta_encoder":
    # XLM-RoBERTa XNLI — model encoder multilingv antrenat pe NLI.
    # Rulat local via HuggingFace Transformers, CPU.
    from transformers import pipeline as hf_pipeline
    _pipe = hf_pipeline(
        "zero-shot-classification",
        model="joeddav/xlm-roberta-large-xnli",
        device=-1,
    )

    def call_model(prompt: str) -> str:
        text = _extract_conversation_text(prompt)
        result = _pipe(text, candidate_labels=LABELS)
        predicted = result["labels"][0]
        score     = result["scores"][0]
        conf = "high" if score > 0.7 else "medium" if score > 0.4 else "low"
        return json.dumps({
            "intent":     predicted,
            "confidence": conf,
            "reasoning":  f"zero-shot score: {score:.4f}",
        })

elif MODEL == "robert_encoder":
    # RoBERT-base — model encoder monolingv (română), antrenat pe MLM.
    # Rulat local via HuggingFace Transformers, CPU.
    from transformers import pipeline as hf_pipeline
    _pipe = hf_pipeline(
        "zero-shot-classification",
        model="readerbench/RoBERT-base",
        device=-1,
    )

    def call_model(prompt: str) -> str:
        text = _extract_conversation_text(prompt)
        result = _pipe(text, candidate_labels=LABELS)
        predicted = result["labels"][0]
        score     = result["scores"][0]
        conf = "high" if score > 0.7 else "medium" if score > 0.4 else "low"
        return json.dumps({
            "intent":    predicted,
            "confidence": conf,
            "reasoning": f"zero-shot score: {score:.4f}",
        })

else:
    raise ValueError(f"Model necunoscut: {MODEL}")

print(f"✓ Model inițializat: {MODEL}")

c:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 393/393 [00:00<00:00, 4510.12it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: joeddav/xlm-roberta-large-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Model inițializat: roberta_encoder


In [9]:
test_prompt = render_prompt(LANG, conversations[0]["turns"], "v4")
raw = call_model(test_prompt)
print(raw)

{"intent": "update_personal_data", "confidence": "medium", "reasoning": "zero-shot score: 0.4645"}


---
## [4] Run experiment
*Rulează modelul pe dataset. Progresul apare linie cu linie.*

In [10]:
results = []
total   = len(conversations)

for i, conv in enumerate(conversations):
    conv_id       = conv["conversation_id"]
    dataset_label = conv["intent"]
    turns         = conv["turns"]

    prompt = render_prompt(LANG, turns, PROMPT_VERSION)

    start = time.monotonic()
    try:
        raw        = call_model(prompt)
        latency_ms = (time.monotonic() - start) * 1000
        intent, confidence, reasoning, parse_failed = parse_response(raw)
    except Exception as e:
        latency_ms   = (time.monotonic() - start) * 1000
        raw          = str(e)
        intent       = None
        confidence   = None
        reasoning    = None
        parse_failed = True

    result = {
        "conversation_id":  conv_id,
        "dataset_label":    dataset_label,
        "model_name":       MODEL,
        "prompt_lang":      LANG,
        "prompt_version":   PROMPT_VERSION,
        "predicted_intent": intent,
        "confidence":       confidence,
        "reasoning":        reasoning,
        "latency_ms":       round(latency_ms, 1),
        "parse_failed":     parse_failed,
        "raw_response":     raw,
    }
    results.append(result)

    # Progress live
    if parse_failed:
        status = "FAIL"
    elif intent == dataset_label:
        status = f"✓  {intent}"
    else:
        status = f"✗  {intent}  (real: {dataset_label})"

    print(f"[{i+1:>3}/{total}]  {conv_id:<25} {status:<50} {latency_ms:.0f}ms")

print(f"\n✓ Terminat. {len(results)} predicții.")

[  1/180]  conv_simple_0001          ✓  update_personal_data                            5719ms
[  2/180]  conv_simple_0002          ✓  update_personal_data                            4016ms
[  3/180]  conv_simple_0003          ✓  update_personal_data                            4328ms
[  4/180]  conv_simple_0004          ✓  update_personal_data                            5390ms
[  5/180]  conv_simple_0005          ✓  update_personal_data                            4829ms
[  6/180]  conv_simple_0006          ✓  close_account                                   5687ms
[  7/180]  conv_simple_0007          ✓  close_account                                   5281ms
[  8/180]  conv_simple_0008          ✗  check_balance  (real: close_account)            5625ms
[  9/180]  conv_simple_0009          ✓  close_account                                   6813ms
[ 10/180]  conv_simple_0010          ✓  close_account                                   5719ms
[ 11/180]  conv_simple_0011          ✓  open_accou

---
## [5] Rezultate
*Metrici sumar + tabel predicții. Nu face niciun API call.*

In [11]:
print(f"{'═'*60}")
print(f"  METRICI SUMAR — {EXP_NAME}")
print(f"{'═'*60}\n")
metrics = print_metrics_summary(results, EXP_NAME)

print(f"\n{'─'*60}")
print(f"  PREDICȚII")
print(f"{'─'*60}\n")
print_predictions_table(results)

print(f"\n{'─'*60}")
print(f"  TOP GREȘELI")
print(f"{'─'*60}\n")
print_top_errors(results)

════════════════════════════════════════════════════════════
  METRICI SUMAR — roberta_encoder__en__v4
════════════════════════════════════════════════════════════

Metrică         Valoare
--------------  -----------------------
Experiment      roberta_encoder__en__v4
Accuracy        53.9%
Macro F1        55.2%
Weighted F1     55.0%
Cohen κ         0.497
Parse failures  0/180  (0.0%)
Latență medie   10644ms
Latență p95     19361ms
N valid         180

────────────────────────────────────────────────────────────
  PREDICȚII
────────────────────────────────────────────────────────────

╭───────────────────┬───────────────────────────────┬─────────────────────────────────────┬──────────────┬───────────╮
│ conv_id           │ dataset_label                 │ predicție                           │ confidence   │ latență   │
├───────────────────┼───────────────────────────────┼─────────────────────────────────────┼──────────────┼───────────┤
│ conv_simple_0001  │ update_personal_data          

---
## [6] Salvare experiment
*Salvează rezultatele în `outputs/exp_{model}__{lang}__{version}.json`.*

In [13]:
payload = {
    "experiment": {
        "name":           EXP_NAME,
        "model":          MODEL,
        "lang":           LANG,
        "prompt_version": PROMPT_VERSION,
        "prompt_file":    f"{LANG}_zero_shot_{PROMPT_VERSION}.jinja",
        "n_conversations":len(results),
        "timestamp":      time.strftime("%Y%m%d_%H%M%S"),
    },
    "metrics": metrics,
    "predictions": results,
}

with open(EXP_FILE, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print(f"✓ Salvat în: {EXP_FILE}")

✓ Salvat în: C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\outputs\exp_roberta_encoder__en__v4.json


---
## [7] Comparație experimente
*Compară două experimente deja salvate — util pentru A/B între versiuni de prompt sau modele diferite.*

Schimbă `EXP_A` și `EXP_B` cu numele experimentelor pe care vrei să le compari.

In [14]:
# ── Setează experimentele de comparat ──────────────────────────────────────
EXP_A = "roberta_encoder__en__v4"
EXP_B = "roberta_encoder__en__v3"
# ──────────────────────────────────────────────────────────────────────────

def load_exp(name: str) -> dict:
    path = OUTPUT_DIR / f"exp_{name}.json"
    with open(path, encoding="utf-8") as f:
        return json.load(f)

exp_a = load_exp(EXP_A)
exp_b = load_exp(EXP_B)

ma = exp_a["metrics"]
mb = exp_b["metrics"]

# ── Metrici side-by-side ───────────────────────────────────────────────────
print(f"{'═'*70}")
print(f"  COMPARAȚIE: {EXP_A}  vs  {EXP_B}")
print(f"{'═'*70}\n")

def d(a, b):
    """Delta formatat cu semn."""
    if a is None or b is None: return "—"
    diff = a - b
    return f"{'+'if diff>=0 else ''}{diff*100:.1f}%"

cmp_rows = [
    ["Accuracy",    f"{ma.get('accuracy',0)*100:.1f}%",    f"{mb.get('accuracy',0)*100:.1f}%",    d(ma.get('accuracy'),    mb.get('accuracy'))],
    ["Macro F1",    f"{ma.get('macro_f1',0)*100:.1f}%",    f"{mb.get('macro_f1',0)*100:.1f}%",    d(ma.get('macro_f1'),    mb.get('macro_f1'))],
    ["Weighted F1", f"{ma.get('weighted_f1',0)*100:.1f}%", f"{mb.get('weighted_f1',0)*100:.1f}%", d(ma.get('weighted_f1'), mb.get('weighted_f1'))],
    ["Cohen κ",     f"{ma.get('kappa',0):.3f}",            f"{mb.get('kappa',0):.3f}",            "—"],
]
print(tabulate(cmp_rows, headers=["Metrică", EXP_A, EXP_B, "Δ (A−B)"], tablefmt="rounded_outline"))

# ── Predicții unde diferă ──────────────────────────────────────────────────
preds_a = {r["conversation_id"]: r for r in exp_a["predictions"]}
preds_b = {r["conversation_id"]: r for r in exp_b["predictions"]}

diff_rows = []
for cid, ra in preds_a.items():
    rb = preds_b.get(cid)
    if rb is None:
        continue
    if ra["predicted_intent"] != rb["predicted_intent"]:
        true_label = ra["dataset_label"]
        ca = "✓" if ra["predicted_intent"] == true_label else "✗"
        cb = "✓" if rb["predicted_intent"] == true_label else "✗"
        diff_rows.append([
            cid, true_label,
            f"{ca} {ra['predicted_intent']}",
            f"{cb} {rb['predicted_intent']}",
        ])

print(f"\n  Conversații unde predicțiile diferă: {len(diff_rows)}")
if diff_rows:
    print(tabulate(diff_rows,
                   headers=["conv_id", "dataset_label", EXP_A, EXP_B],
                   tablefmt="simple"))

══════════════════════════════════════════════════════════════════════
  COMPARAȚIE: roberta_encoder__en__v4  vs  roberta_encoder__en__v3
══════════════════════════════════════════════════════════════════════

╭─────────────┬───────────────────────────┬───────────────────────────┬───────────╮
│ Metrică     │ roberta_encoder__en__v4   │ roberta_encoder__en__v3   │ Δ (A−B)   │
├─────────────┼───────────────────────────┼───────────────────────────┼───────────┤
│ Accuracy    │ 53.9%                     │ 53.9%                     │ +0.0%     │
│ Macro F1    │ 55.2%                     │ 55.2%                     │ +0.0%     │
│ Weighted F1 │ 55.0%                     │ 55.0%                     │ +0.0%     │
│ Cohen κ     │ 0.497                     │ 0.497                     │ —         │
╰─────────────┴───────────────────────────┴───────────────────────────┴───────────╯

  Conversații unde predicțiile diferă: 0


In [14]:
test_prompt = render_prompt("ro", conversations[0]["turns"], "v3")
# Simulează extracția din RoBERTa
lines = test_prompt.split("\n")
conv_lines, in_conv = [], False
for line in lines:
    if line.strip().startswith("## Convers"):
        in_conv = True; continue
    if in_conv and line.strip().startswith("##"):
        break
    if in_conv and line.strip():
        conv_lines.append(line.strip())
print(" ".join(conv_lines))

USER: Bună, aș dori să actualizez numărul meu de telefon. ASSISTANT: Bună! Sunt bucuros să vă ajut. Pentru a actualiza numărul de telefon, vă voi solicita să confirmați identitatea. Care este numărul dumneavoastră de cont? USER: Contul meu este 1234567890. ASSISTANT: Mulțumesc. Care este noul număr de telefon pe care doriți să-l înregistrați? USER: Noul număr este 0721234567. ASSISTANT: Perfect. Am actualizat numărul dumneavoastră de telefon la 0721234567. Veți primi o confirmare prin SMS în scurt timp. # Instrucțiuni - Clasifică intenția pe baza **scopului** cu care utilizatorul a inițiat conversația, de obicei definită în primele replici. - Dacă utilizatorul adresează **orice solicitare** în afara domeniului bancar în orice moment al conversației, folosește `fallback`. - Replicile ulterioare, precum confirmări, furnizare de date sau „mulțumesc”, nu schimbă intenția. # Planificare și verificare - Identifică scopul principal urmărit de utilizator. - Verifică dacă există vreo solicitare

In [16]:
from collections import defaultdict
from tabulate import tabulate
import json

# ── Schimbă aici ce experiment vrei să analizezi ──
EXP_TO_ANALYZE = "gemini_2.5_flash__ro__v4"

with open(OUTPUT_DIR / f"exp_{EXP_TO_ANALYZE}.json", encoding="utf-8") as f:
    exp = json.load(f)

predictions = exp["predictions"]

# ── Toate greșelile: clasa reală, clasa prezisă, conv_id ──
print("═" * 70)
print(f"  GREȘELI — {EXP_TO_ANALYZE}")
print("═" * 70)

rows = []
for p in predictions:
    if p["parse_failed"] or p["predicted_intent"] is None:
        continue
    if p["predicted_intent"] != p["dataset_label"]:
        rows.append([
            p["conversation_id"],
            p["dataset_label"],
            p["predicted_intent"],
        ])

rows.sort(key=lambda r: r[1])  # sortat după clasa reală
print(tabulate(rows,
               headers=["conv_id", "Clasa reală", "Prezis ca"],
               tablefmt="rounded_outline"))

print(f"\n  Total greșeli: {len(rows)} / {len(predictions)}")

══════════════════════════════════════════════════════════════════════
  GREȘELI — gemini_2.5_flash__ro__v4
══════════════════════════════════════════════════════════════════════
╭───────────────────┬──────────────────────┬───────────────────────────────╮
│ conv_id           │ Clasa reală          │ Prezis ca                     │
├───────────────────┼──────────────────────┼───────────────────────────────┤
│ conv_simple_0062  │ fallback             │ report_suspicious_transaction │
│ conv_simple_0057  │ general_product_info │ schedule_advisor_meeting      │
│ conv_complex_0026 │ general_product_info │ schedule_advisor_meeting      │
│ conv_simple_0076  │ general_product_info │ open_account                  │
╰───────────────────┴──────────────────────┴───────────────────────────────╯

  Total greșeli: 4 / 180


In [17]:
import json
from pathlib import Path
from collections import defaultdict
from tabulate import tabulate

OUTPUT_DIR = Path(r"C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\outputs")

# Încarcă toate experimentele
experiments = {}
for f in sorted(OUTPUT_DIR.glob("exp_*.json")):
    with open(f, encoding="utf-8") as fp:
        data = json.load(fp)
    name = data["experiment"]["name"]
    experiments[name] = data

# Sumar metrici per experiment
print("═" * 80)
print("  SUMAR METRICI — toate experimentele")
print("═" * 80)

rows = []
for name, data in sorted(experiments.items()):
    m = data.get("metrics", {})
    rows.append([
        name,
        f"{m.get('accuracy', 0)*100:.1f}%",
        f"{m.get('macro_f1', 0)*100:.1f}%",
        f"{m.get('n_failed', 0)}/{m.get('n_valid', 0)+m.get('n_failed', 0)}",
    ])

print(tabulate(rows,
               headers=["Experiment", "Accuracy", "Macro F1", "Parse fails"],
               tablefmt="rounded_outline"))

════════════════════════════════════════════════════════════════════════════════
  SUMAR METRICI — toate experimentele
════════════════════════════════════════════════════════════════════════════════
╭────────────────────────────────┬────────────┬────────────┬───────────────╮
│ Experiment                     │ Accuracy   │ Macro F1   │ Parse fails   │
├────────────────────────────────┼────────────┼────────────┼───────────────┤
│ aya_expanse_8b__en__v1         │ 85.6%      │ 84.2%      │ 0/180         │
│ aya_expanse_8b__en__v2         │ 85.6%      │ 84.5%      │ 0/180         │
│ aya_expanse_8b__en__v3         │ 88.3%      │ 87.2%      │ 0/180         │
│ aya_expanse_8b__en__v4         │ 91.1%      │ 90.9%      │ 0/180         │
│ aya_expanse_8b__ro__v1         │ 87.8%      │ 86.3%      │ 0/180         │
│ aya_expanse_8b__ro__v2         │ 90.0%      │ 88.9%      │ 0/180         │
│ aya_expanse_8b__ro__v3         │ 90.0%      │ 89.0%      │ 0/180         │
│ aya_expanse_8b__ro__v4      

In [18]:
import json
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score

OUTPUT_DIR = Path(r"C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\outputs")

for exp_file in sorted(OUTPUT_DIR.glob("exp_*.json")):
    with open(exp_file, encoding="utf-8") as f:
        data = json.load(f)

    predictions = data.get("predictions", [])
    y_true = [r["dataset_label"]    for r in predictions if not r["parse_failed"] and r["predicted_intent"]]
    y_pred = [r["predicted_intent"] for r in predictions if not r["parse_failed"] and r["predicted_intent"]]
    fails  = sum(1 for r in predictions if r["parse_failed"])

    if y_true:
        data["metrics"] = {
            "accuracy":     round(accuracy_score(y_true, y_pred), 4),
            "macro_f1":     round(f1_score(y_true, y_pred, average="macro",    labels=LABELS, zero_division=0), 4),
            "weighted_f1":  round(f1_score(y_true, y_pred, average="weighted", labels=LABELS, zero_division=0), 4),
            "cohen_kappa":  round(cohen_kappa_score(y_true, y_pred, labels=LABELS), 4),
            "n_valid":      len(y_true),
            "n_failed":     fails,
        }

    with open(exp_file, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    m = data["metrics"]
    print(f"✓ {exp_file.name:<45} acc={m['accuracy']*100:.1f}%  f1={m['macro_f1']*100:.1f}%")

✓ exp_aya_expanse_8b__en__v1.json               acc=85.6%  f1=84.2%
✓ exp_aya_expanse_8b__en__v2.json               acc=85.6%  f1=84.5%
✓ exp_aya_expanse_8b__en__v3.json               acc=88.3%  f1=87.2%
✓ exp_aya_expanse_8b__en__v4.json               acc=91.1%  f1=90.9%
✓ exp_aya_expanse_8b__ro__v1.json               acc=87.8%  f1=86.3%
✓ exp_aya_expanse_8b__ro__v2.json               acc=90.0%  f1=88.9%
✓ exp_aya_expanse_8b__ro__v3.json               acc=90.0%  f1=89.0%
✓ exp_aya_expanse_8b__ro__v4.json               acc=90.0%  f1=89.8%
✓ exp_gemini_2.5_flash__en__v1.json             acc=95.6%  f1=95.4%
✓ exp_gemini_2.5_flash__en__v2.json             acc=97.2%  f1=97.2%
✓ exp_gemini_2.5_flash__en__v3.json             acc=97.2%  f1=97.2%
✓ exp_gemini_2.5_flash__en__v4.json             acc=97.2%  f1=97.2%
✓ exp_gemini_2.5_flash__ro__v1.json             acc=95.0%  f1=94.7%
✓ exp_gemini_2.5_flash__ro__v2.json             acc=97.2%  f1=97.2%
✓ exp_gemini_2.5_flash__ro__v3.json             